# HGND Reconstruction GNN — Preprocessing & DataLoader

This notebook uses a **disk-based PyG `Dataset`** (`HGNDGraphDataset`) that:
- Loads CSV hit/MC-truth files and performs all feature engineering
- Constructs heterogeneous graphs (HeteroData) in parallel via `multiprocessing`
- Serialises graphs in **shards** (default 1024 graphs per `.pt` file) — with ~2M events this produces ~2k shard files instead of 2M individual files
- Provides lazy loading through `__getitem__` with an LRU shard cache → low memory footprint

The base module lives in `HGNDRecoGNN/data/graph_dataset.py`.

In [ ]:
import os
import sys
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

import torch
import torch.nn.functional as F
import torch_geometric
from torch_geometric.loader import DataLoader
from torch_geometric.data import HeteroData
from torch.utils.data import random_split

from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# sys.path needs the directory that *contains* the HGNDRecoGNN package.
# cwd is .../HGNDRecoGNN/notebooks  →  ../.. = parent directory containing the package
PACKAGE_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if PACKAGE_ROOT not in sys.path:
    sys.path.insert(0, PACKAGE_ROOT)

# Also keep a convenience reference to the HGNDRecoGNN directory
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))

print(f'PyTorch {torch.__version__}')
print(f'PyG    {torch_geometric.__version__}')
print(f'Package root (on sys.path): {PACKAGE_ROOT}')
print(f'Project root:               {PROJECT_ROOT}')

## 1. Import the base dataloader module

The module `HGNDRecoGNN.data.graph_dataset` contains:
- `HGNDGraphDataset` — disk-based PyG `Dataset` with parallel `process()`
- `load_hits` — CSV loading + feature engineering
- `prepare_halves` — top/bot split, scaler fitting
- `_build_single_graph` — per-event graph construction (picklable, used by multiprocessing)

In [ ]:
from HGNDRecoGNN.data.graph_dataset import (
    HGNDGraphDataset,
    FEATURES,
    load_hits,
    prepare_halves,
    _build_single_graph,
)
print('Module imported successfully')
print(f'Features: {FEATURES}')

## 2. Configure dataset parameters

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────────
HITS_CSV_DIR = os.path.join(PROJECT_ROOT, 'data', 'smash_xecs_2.87gev_hardSkyrme_defaultSpot')
DATASET_ROOT = os.path.join(os.getcwd(), 'cache', 'ndet_dataset_smash_defaultSpot')

# ── Graph construction parameters ─────────────────────────────────────────
RLOCAL   = 3.6    # spatial radius (LayerId, RowId, ColumnId space)
TWINDOW  = 1.5    # temporal radius (fTime)

# ── Event / shard limits (set to None to use everything) ──────────────────
# MAX_EVENTS : cap events processed per half (top/bot) — triggers a new build
#              if the dataset doesn't exist yet or FORCE_REBUILD=True.
#              e.g. MAX_EVENTS=500 → ≤ 1 000 graphs total.
MAX_EVENTS = 100000  # ← set to e.g. 500 to limit processing

# NUM_SHARDS : load only the first N already-processed shard files.
#              NO processing is triggered — the dataset must already be on disk.
#              Each shard holds SHARD_SIZE graphs (default 1024), so
#              NUM_SHARDS=2 → up to 2 048 graphs.  Fast way to test
#              downstream code (DataLoader, model, training loop).
NUM_SHARDS = 100     # ← set to None to load all shards

# ── Serialisation ─────────────────────────────────────────────────────────
SHARD_SIZE = 1024  # graphs per shard file (2M graphs → ~2k files)

# ── Parallelisation ───────────────────────────────────────────────────────
NUM_WORKERS_PROCESS = 8   # workers for graph construction (multiprocessing)
NUM_WORKERS_LOADER  = 0   # workers for DataLoader (0 = main process on macOS)

# ── Training ──────────────────────────────────────────────────────────────
BATCH_SIZE = 512
SEED = 42

# ── Device ────────────────────────────────────────────────────────────────
if torch.backends.mps.is_available():
    mps_device = torch.device('mps')
    print(f'Using MPS device')
else:
    mps_device = torch.device('cpu')
    print('MPS not available, falling back to CPU')

torch.manual_seed(SEED)
np.random.seed(SEED)

print(f'\nCSV dir:      {HITS_CSV_DIR}')
print(f'Dataset root: {DATASET_ROOT}')
print(f'rlocal={RLOCAL}, twindow={TWINDOW}, shard_size={SHARD_SIZE}, workers={NUM_WORKERS_PROCESS}')
print(f'max_events:   {MAX_EVENTS}  (None = all events, used during processing)')
print(f'num_shards:   {NUM_SHARDS}  (None = all shards, used when loading pre-processed data)')


## 3. Create the dataset (parallel graph construction + sharded serialisation)

Processing works in **chunks of `shard_size`** events at a time — only one chunk is in memory.

**Checkpointing:** after each shard is written, a `_progress.json` file records the state.
If the kernel crashes, re-running this cell will **resume from the last completed shard**.

To **force a full rebuild**, set `FORCE_REBUILD = True` below.

In [ ]:
import shutil, importlib
import HGNDRecoGNN.data.graph_dataset as _gd
importlib.reload(_gd)                        # pick up any module changes
HGNDGraphDataset = _gd.HGNDGraphDataset

FORCE_REBUILD = False   # ← set True to wipe all shards and start from scratch

processed_dir = os.path.join(DATASET_ROOT, 'processed')
meta_path     = os.path.join(processed_dir, 'meta.json')
progress_path = os.path.join(processed_dir, '_progress.json')

if FORCE_REBUILD and os.path.exists(processed_dir):
    print(f'FORCE_REBUILD: removing {processed_dir} …')
    shutil.rmtree(processed_dir)
elif os.path.exists(meta_path):
    import json as _json
    with open(meta_path) as _f:
        _m = _json.load(_f)
    _total = _m['num_graphs']
    _exposed = min(_m['num_shards'], NUM_SHARDS) * _m['shard_size'] if NUM_SHARDS else _total
    print(f'Existing dataset: {_total} graphs in {_m["num_shards"]} shards'
          + (f'  →  loading first {NUM_SHARDS} shard(s) (~{min(_exposed, _total)} graphs)'
             if NUM_SHARDS else ''))
elif os.path.exists(progress_path):
    import json as _json
    with open(progress_path) as _f:
        _p = _json.load(_f)
    _pshards = _p.get('shard_idx', 0)
    _pgraphs = _p.get('graph_idx', 0)
    if NUM_SHARDS:
        print(f'Incomplete build detected: {_pgraphs} graphs in {_pshards} shards on disk '
              f'(no meta.json).  Loading first {NUM_SHARDS} shard(s) — set NUM_SHARDS=None '
              f'or FORCE_REBUILD=True to resume/rebuild.')
    else:
        # No num_shards cap and no meta.json → resume processing
        if os.path.exists(meta_path):
            os.remove(meta_path)
        print(f'Incomplete build ({_pgraphs} graphs, {_pshards} shards) — resuming …')

t0 = time.time()

dataset = HGNDGraphDataset(
    root=DATASET_ROOT,
    hits_csv_dir=HITS_CSV_DIR,
    rlocal=RLOCAL,
    twindow=TWINDOW,
    num_workers=NUM_WORKERS_PROCESS,
    shard_size=SHARD_SIZE,
    device=None,        # keep on CPU; move to MPS at training time
    max_events=MAX_EVENTS,   # cap events during processing (None = all)
    num_shards=NUM_SHARDS,   # cap shards when loading pre-processed data (None = all)
)

elapsed = time.time() - t0
meta = dataset._load_meta()
print(f'\nDataset ready: {len(dataset)} graphs  '
      f'(shards on disk: {meta["num_shards"]}, exposed: {NUM_SHARDS if NUM_SHARDS else meta["num_shards"]})  '
      f'({elapsed:.1f}s)')

# ── Preload all shards into memory ────────────────────────────────────────
# This eliminates the per-batch torch.load() bottleneck during training.
# 100 shards × ~5 MB ≈ 540 MB RAM — well within reach.
dataset.preload()

## 4. Inspect sample graphs

In [ ]:
# Look at one graph
g = dataset[0]
print(g)
print(f'\nhits:     {g["hits"].x.shape[0]} nodes, {g["hits"].x.shape[1]} features')
print(f'edges:    {g["hits","hits"].edge_index.shape[1]}')
print(f'clusters: {g["clusters"].x.shape[0]}')
print(f'istop:    {g.istop.item()}')
print(f'Row:      {g.Row.item()}')
print(f'evtlabel: {g.evtlabel.item()}')

In [ ]:
# Summary statistics across first 100 graphs
n_hits_list, n_edges_list, n_clusters_list = [], [], []
for i in range(min(100, len(dataset))):
    g = dataset[i]
    n_hits_list.append(g['hits'].x.shape[0])
    n_edges_list.append(g['hits', 'hits'].edge_index.shape[1])
    n_clusters_list.append(g['clusters'].x.shape[0])

fig, axes = plt.subplots(1, 3, figsize=(14, 3))
axes[0].hist(n_hits_list, bins=30);    axes[0].set_title('Hits per event')
axes[1].hist(n_edges_list, bins=30);   axes[1].set_title('Edges per event')
axes[2].hist(n_clusters_list, bins=30); axes[2].set_title('Clusters per event')
plt.tight_layout()
plt.show()

## 5. Verify data integrity (round-trip check)

In [ ]:
# Reload shard 0 directly from disk and compare with dataset[0]
g_mem  = dataset[0]
shard0 = torch.load(
    os.path.join(dataset.processed_dir, 'shard_0.pt'),
    weights_only=False,
)
g_disk = shard0[0]

assert torch.equal(g_mem['hits'].x, g_disk['hits'].x), 'hits.x mismatch!'
assert torch.equal(g_mem['hits', 'hits'].edge_index,
                   g_disk['hits', 'hits'].edge_index), 'edge_index mismatch!'
assert torch.equal(g_mem['clusters'].y, g_disk['clusters'].y), 'clusters.y mismatch!'
assert torch.equal(g_mem.true_links, g_disk.true_links), 'true_links mismatch!'
print(f'✓ Round-trip integrity check passed  (shard 0 has {len(shard0)} graphs)')

## 6. Train / test split & DataLoader

In [ ]:
torch.manual_seed(SEED)

n_total = len(dataset)
n_train = n_total // 2
n_test  = n_total - n_train

train_ds, test_ds = random_split(dataset, [n_train, n_test])
print(f'Train: {len(train_ds)},  Test: {len(test_ds)}')

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS_LOADER)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS_LOADER)

In [ ]:
# Inspect one batch & profile DataLoader throughput
batch = next(iter(train_loader))
print(batch)
print(f'\nBatch hit nodes:     {batch["hits"].x.shape}')
print(f'Batch hit edges:     {batch["hits","hits"].edge_index.shape}')
print(f'Batch cluster nodes: {batch["clusters"].x.shape}')
print(f'Batch cluster edges: {batch["clusters","clusters"].edge_index.shape}')
print(f'Num graphs in batch: {batch.batch_dict["hits"].max().item() + 1}')

# ── DataLoader throughput benchmark ──────────────────────────────────────
print('\n--- DataLoader throughput benchmark (train_loader) ---')
n_batches_bench = min(20, len(train_loader))
t0 = time.time()
for i, _b in enumerate(train_loader):
    if i >= n_batches_bench:
        break
elapsed = time.time() - t0
graphs_loaded = n_batches_bench * BATCH_SIZE
print(f'{n_batches_bench} batches in {elapsed:.2f}s  '
      f'({elapsed/n_batches_bench*1000:.0f} ms/batch, '
      f'{graphs_loaded/elapsed:.0f} graphs/s)')
print(f'Preloaded: {dataset.is_preloaded}')

## 7. Model definition

Same `Net` architecture: MPS hit-branch + CPU cluster-branch (DynamicEdgeConv is CPU-only).

In [ ]:
from torch_geometric.nn import (
    GCNConv, EdgeConv, GraphConv, DynamicEdgeConv, SAGEConv, BatchNorm, Sequential,
)
from torch_geometric.nn.pool import avg_pool_x
from torch.nn import Linear, ReLU, BatchNorm1d as BN
from torch.nn import Sequential as Seq, Linear as Lin


class Net(torch.nn.Module):
    def __init__(self, hidden_channels, num_layers, num_features):
        super(Net, self).__init__()
        self.convs = torch.nn.ModuleList()
        self.batch_norms = torch.nn.ModuleList()

        nn = Seq(Lin(num_features * 2, 64), ReLU(), Lin(64, 64), ReLU(), Lin(64, 64), ReLU())
        self.convs.append(EdgeConv(nn, aggr='sum'))

        nn = Seq(Lin(128, 128), ReLU(), Lin(128, 128), ReLU(), Lin(128, 256), ReLU())
        self.convs.append(EdgeConv(nn, aggr='sum'))

        for _ in range(num_layers - 4):
            self.convs.append(SAGEConv(-1, hidden_channels))
            self.batch_norms.append(BatchNorm(hidden_channels))
        self.convs.append(GraphConv(hidden_channels, hidden_channels))
        self.convs.append(GraphConv(hidden_channels, hidden_channels))

        # Hit-level edge scoring (MPS)
        self.edge_out = Sequential('x', [
            (Linear(hidden_channels * 2, hidden_channels), 'x -> x'),
            ReLU(inplace=True),
            Linear(hidden_channels, hidden_channels),
            ReLU(inplace=True),
            Linear(hidden_channels, 1),
        ])

        self.hitcl_out = Sequential('x', [
            (Linear(hidden_channels, hidden_channels // 2), 'x -> x'),
            ReLU(inplace=True),
            Linear(hidden_channels // 2, hidden_channels // 2),
            ReLU(inplace=True),
            Linear(hidden_channels // 2, 1),
        ])

        # Cluster-level modules — permanently CPU (DynamicEdgeConv/knn is CPU-only)
        nn_cl = Seq(Lin(hidden_channels * 2, hidden_channels * 4), ReLU(),
                    Lin(hidden_channels * 4, hidden_channels * 4), ReLU(),
                    Lin(hidden_channels * 4, hidden_channels), ReLU())
        self.cluster_conv_cpu = DynamicEdgeConv(nn_cl, k=100, aggr='sum')

        self.clclass_out_cpu = Sequential('x', [
            (Linear(hidden_channels, hidden_channels // 2), 'x -> x'),
            ReLU(inplace=True),
            Linear(hidden_channels // 2, hidden_channels // 2),
            ReLU(inplace=True),
            Linear(hidden_channels // 2, 1),
        ])

        self.clenergy_out_cpu = Sequential('x', [
            (Linear(hidden_channels, hidden_channels // 2), 'x -> x'),
            ReLU(inplace=True),
            Linear(hidden_channels // 2, hidden_channels // 2),
            ReLU(inplace=True),
            Linear(hidden_channels // 2, 1),
        ])

        self.cl_edge_out_cpu = Sequential('x', [
            (Linear(hidden_channels * 2, hidden_channels), 'x -> x'),
            ReLU(inplace=True),
            Linear(hidden_channels, hidden_channels),
            ReLU(inplace=True),
            Linear(hidden_channels, 1),
        ])

    def to(self, device):
        """Override to keep *_cpu modules on CPU after device transfer."""
        cpu_state = {
            'cluster_conv_cpu': self.cluster_conv_cpu.state_dict(),
            'clclass_out_cpu':  self.clclass_out_cpu.state_dict(),
            'clenergy_out_cpu': self.clenergy_out_cpu.state_dict(),
            'cl_edge_out_cpu':  self.cl_edge_out_cpu.state_dict(),
        }
        super().to(device)
        self.cluster_conv_cpu.load_state_dict(cpu_state['cluster_conv_cpu'])
        self.clclass_out_cpu.load_state_dict(cpu_state['clclass_out_cpu'])
        self.clenergy_out_cpu.load_state_dict(cpu_state['clenergy_out_cpu'])
        self.cl_edge_out_cpu.load_state_dict(cpu_state['cl_edge_out_cpu'])
        self.cluster_conv_cpu.cpu()
        self.clclass_out_cpu.cpu()
        self.clenergy_out_cpu.cpu()
        self.cl_edge_out_cpu.cpu()
        return self

    def forward(self, x, edge_index, edge_index_cl, clusters, batch):
        # EdgeConv layers (no batch norm)
        x = self.convs[0](x, edge_index).relu()
        x = self.convs[1](x, edge_index).relu()

        # SAGEConv layers with batch norm
        for conv, bn in zip(self.convs[2:-2], self.batch_norms):
            x = bn(conv(x, edge_index)).relu()

        row, col = edge_index
        new_edge_attr = self.edge_out(torch.cat([x[row], x[col]], dim=-1))
        new_edge_attr = torch.sigmoid(new_edge_attr).squeeze(1)

        x = self.convs[-2](x, edge_index, new_edge_attr)
        x = x.relu()
        x = self.convs[-1](x, edge_index)
        x = x.relu()

        # Cluster branch on CPU (avg_pool_x + DynamicEdgeConv are CPU-only)
        cl_x, cl_batch = avg_pool_x(clusters.cpu(), x.detach().cpu(), batch.cpu(),
                                     batch_size=BATCH_SIZE)
        cl_x = self.cluster_conv_cpu(cl_x, cl_batch)

        cl_cl = self.clclass_out_cpu(cl_x).sigmoid()
        cl_e  = self.clenergy_out_cpu(cl_x)

        row_cl, col_cl = edge_index_cl.cpu()
        cl_connection = self.cl_edge_out_cpu(
            torch.cat([cl_x[row_cl], cl_x[col_cl]], dim=-1)
        )
        cl_connection = torch.sigmoid(cl_connection).squeeze(1)

        x = self.hitcl_out(x)
        x = torch.sigmoid(x)

        _dev = x.device
        return (new_edge_attr, x.squeeze(1),
                cl_cl.squeeze(1).to(_dev), cl_e.squeeze(1).to(_dev),
                cl_connection.to(_dev), cl_batch.to(_dev))

## 8. Training setup

In [ ]:
# Hyperparameters
reg         = 1e-5
epochs      = 20
num_hidden  = 512
num_layers  = 8
num_features = dataset[0]['hits'].num_features

model = Net(num_hidden, num_layers, num_features).to(mps_device)

# Materialise lazy (SAGEConv -1) parameters with a dummy forward pass
with torch.no_grad():
    _g0 = dataset[0].clone().to(mps_device)
    _cl0 = _g0.cluster.squeeze(-1) + 1
    _b0  = torch.zeros(_g0['hits'].x.size(0), dtype=torch.long, device=mps_device)
    model.eval()
    model(
        _g0['hits'].x,
        _g0['hits', 'hits'].edge_index,
        _g0['clusters', 'clusters'].edge_index,
        _cl0, _b0,
    )
    del _g0, _cl0, _b0
model.train()

bce = torch.nn.BCELoss(reduction='mean')
mse = torch.nn.MSELoss(reduction='mean')

optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=reg)
scheduler = torch.optim.lr_scheduler.MultiStepLR(optimizer, milestones=[10, 15, 25], gamma=0.1)

CHECKPOINT_PATH = 'checkpoints/Hitcl_cluster_surface.pt'
os.makedirs('checkpoints', exist_ok=True)

train_losses = []
test_losses  = []
minloss      = 1e10

print(f'Model params: {sum(p.numel() for p in model.parameters()):,}')
print(f'num_features: {num_features}')

## 9. Training & evaluation loop

In [ ]:
from IPython.display import clear_output


def _reindex_clusters(graph):
    """Re-index per-graph cluster ids into a contiguous batch-global range."""
    batched_cluster = torch.zeros_like(graph.cluster.squeeze(-1))
    n_cl_prev = 0
    for i in range(graph.batch_dict['hits'].max().item() + 1):
        mask = (graph.batch_dict['hits'] == i)
        batched_cluster[mask] = graph.cluster.squeeze(-1)[mask] + n_cl_prev + 1
        n_cl_prev = batched_cluster.max().item()
    return batched_cluster


def train_epoch():
    model.train()
    losses = []
    for graph in tqdm(train_loader, desc='train'):
        graph = graph.to(mps_device)
        model.zero_grad()
        batched_cluster = _reindex_clusters(graph)

        link_scores, outhit, outcl, oute, link_cl, cl_batch = model(
            graph['hits'].x,
            graph['hits', 'hits'].edge_index,
            graph['clusters', 'clusters'].edge_index,
            batched_cluster,
            graph.batch_dict['hits'],
        )

        hitloss  = bce(outhit, graph['hits'].y.float())
        linkloss = bce(link_scores, graph.true_links.float())

        cl_mask = graph.cllabel.bool()
        clusterloss = (bce(outcl, graph.cllabel.float())
                       + mse(oute[cl_mask], graph.clenergy[cl_mask].float()))

        conn_mask = graph['clusters'].connection_mask.bool()
        linkloss_cl = bce(link_cl[conn_mask],
                          graph['clusters'].y[conn_mask].float())

        loss = hitloss + linkloss + clusterloss + linkloss_cl
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        losses.append(loss.item())
    return np.mean(losses)


@torch.no_grad()
def test_epoch():
    model.eval()
    losses = []
    for graph in tqdm(test_loader, desc='test'):
        graph = graph.to(mps_device)
        batched_cluster = _reindex_clusters(graph)

        link_scores, outhit, outcl, oute, link_cl, cl_batch = model(
            graph['hits'].x,
            graph['hits', 'hits'].edge_index,
            graph['clusters', 'clusters'].edge_index,
            batched_cluster,
            graph.batch_dict['hits'],
        )

        hitloss  = bce(outhit, graph['hits'].y.float())
        linkloss = bce(link_scores, graph.true_links.float())

        cl_mask = graph.cllabel.bool()
        clusterloss = (bce(outcl, graph.cllabel.float())
                       + mse(oute[cl_mask], graph.clenergy[cl_mask].float()))

        conn_mask = graph['clusters'].connection_mask.bool()
        linkloss_cl = bce(link_cl[conn_mask],
                          graph['clusters'].y[conn_mask].float())

        loss = hitloss + linkloss + clusterloss + linkloss_cl
        losses.append(loss.item())
    return np.mean(losses)


for epoch in range(epochs):
    loss = train_epoch()
    scheduler.step()
    train_losses.append(loss)

    test_loss = test_epoch()
    test_losses.append(test_loss)

    clear_output(wait=True)
    print(f'Epoch {epoch:03d}  Loss: {loss:.4f}  Val: {test_loss:.4f}  '
          f'lr={optimizer.param_groups[0]["lr"]:.5f}')

    if loss < minloss:
        minloss = loss
        torch.save(model, CHECKPOINT_PATH)

    plt.figure(figsize=(8, 4))
    plt.plot(train_losses, label='Train')
    plt.plot(test_losses, label='Test')
    plt.xlabel('Epoch'); plt.ylabel('Loss')
    plt.legend(); plt.grid(True)
    plt.tight_layout(); plt.show()

## 10. Prediction & evaluation export

Run the trained model on the test set and save per-hit and per-cluster DataFrames
for downstream analysis (ROC curves, energy resolution, etc.).

In [ ]:
RESULTS_DIR = os.path.join(os.getcwd(), 'results')
os.makedirs(RESULTS_DIR, exist_ok=True)

# ── Accumulators (hit-level) ──────────────────────────────────────────────
all_hit_rows   = []   # event Row id per hit
all_hit_inst   = []   # within-event hit index
all_hit_cl     = []   # cluster id per hit
all_hit_score  = []   # model neutron-hit score
all_hit_label  = []   # true neutron label (0/1)
all_hit_istop  = []   # top (1) or bottom (0) half

# ── Accumulators (edge-level) ─────────────────────────────────────────────
all_link_score = []
all_link_label = []

# ── Accumulators (cluster-level) ──────────────────────────────────────────
all_cl_rows    = []   # event Row id per cluster
all_cl_id      = []   # cluster id
all_cl_score   = []   # model cluster score
all_cl_label   = []   # true cluster label
all_cl_epred   = []   # predicted energy
all_cl_etrue   = []   # true energy
all_cl_istop   = []

model.eval()
t0 = time.time()

with torch.no_grad():
    for graph in tqdm(test_loader, desc='predict'):
        graph = graph.to(mps_device)
        batched_cluster = _reindex_clusters(graph)

        link_scores, outhit, outcl, oute, link_cl, cl_batch = model(
            graph['hits'].x,
            graph['hits', 'hits'].edge_index,
            graph['clusters', 'clusters'].edge_index,
            batched_cluster,
            graph.batch_dict['hits'],
        )

        hit_batch = graph.batch_dict['hits']
        n_graphs  = hit_batch.max().item() + 1

        # Pre-compute per-graph hit counts (vectorised)
        _, hit_counts = hit_batch.unique(return_counts=True)

        # Hit-level: scatter Row and istop to every hit
        for gi in range(n_graphs):
            n_h = hit_counts[gi].item()
            all_hit_rows.extend([graph.Row[gi].item()] * n_h)
            all_hit_istop.extend([graph.istop[gi].item()] * n_h)
            all_hit_inst.extend(range(n_h))

        all_hit_cl.extend(graph.cluster.squeeze(-1).cpu().tolist())
        all_hit_label.extend(graph['hits'].y.cpu().long().tolist())
        all_hit_score.extend(outhit.cpu().tolist())

        # Edge-level
        all_link_label.extend(graph.true_links.cpu().long().tolist())
        all_link_score.extend(link_scores.cpu().tolist())

        # Cluster-level: scatter Row and istop to every cluster
        _, cl_counts = cl_batch.unique(return_counts=True)
        for gi in range(n_graphs):
            n_c = cl_counts[gi].item()
            all_cl_rows.extend([graph.Row[gi].item()] * n_c)
            all_cl_istop.extend([graph.istop[gi].item()] * n_c)
            all_cl_id.extend(range(n_c))

        all_cl_label.extend(graph.cllabel.cpu().long().tolist())
        all_cl_score.extend(outcl.cpu().tolist())
        all_cl_etrue.extend(graph.clenergy.cpu().float().tolist())
        all_cl_epred.extend(oute.cpu().tolist())

elapsed = time.time() - t0
print(f'Prediction done: {len(all_hit_score)} hits, '
      f'{len(all_cl_score)} clusters, {len(all_link_score)} edges  '
      f'({elapsed:.1f}s)')

# ── Save results ──────────────────────────────────────────────────────────
hits_df = pd.DataFrame({
    'Row':       all_hit_rows,
    'Instance':  all_hit_inst,
    'ClusterID': all_hit_cl,
    'score':     all_hit_score,
    'label':     all_hit_label,
    'istop':     all_hit_istop,
}).astype({'Row': int, 'Instance': int, 'ClusterID': int, 'istop': int})

clusters_df = pd.DataFrame({
    'Row':       all_cl_rows,
    'ClusterID': all_cl_id,
    'cl_score':  all_cl_score,
    'e_pred':    all_cl_epred,
    'cl_istop':  all_cl_istop,
    'cl_label':  all_cl_label,
    'e_true':    all_cl_etrue,
}).astype({'Row': int, 'ClusterID': int, 'cl_istop': int, 'cl_label': int})

edges_df = pd.DataFrame({
    'link_score': all_link_score,
    'link_label': all_link_label,
})

hits_path = os.path.join(RESULTS_DIR, 'pred_hits_smash.pkl')
cl_path   = os.path.join(RESULTS_DIR, 'pred_clusters_smash.pkl')
edges_path = os.path.join(RESULTS_DIR, 'pred_edges_smash.pkl')

hits_df.to_pickle(hits_path)
clusters_df.to_pickle(cl_path)
edges_df.to_pickle(edges_path)

print(f'\nSaved:')
print(f'  Hits:     {hits_path}  ({len(hits_df):,} rows)')
print(f'  Clusters: {cl_path}  ({len(clusters_df):,} rows)')
print(f'  Edges:    {edges_path}  ({len(edges_df):,} rows)')
print(f'\nhits_df.head():')
hits_df.head()